# JOB4 SESSION 0 — PRETRAIN + EMBED SHARD 0
Preconfigured: `RUN_PRETRAIN=True`, `SHARD=0`, `NSHARDS=3`. Use A100 if available. Run all cells; do not run pretraining in any other session.
Monitor: panel extraction -> epochs 0..19 -> `encoder_final.pt` -> shard-0 embedding -> shard saved or `ALL SHARDS COMBINED`. Epoch checkpoints and embedding parts resume from Drive. Runtime unassigns after its own shard completes.

In [ ]:
RUN_PRETRAIN = True
SHARD = 0
NSHARDS = 3
EMBED_BATCH = 512
WORKERS = 4
DRIVE_PANELS = "/content/drive/MyDrive/brain_packs/panel_farm"
OUT_DIR = "/content/drive/MyDrive/brain_packs/panel_embed"
RAW = "https://raw.githubusercontent.com/vicjayjay/brain-packs/main/code/"
print("SESSION 0: pretrain + embed shard 0/3; host-RAM-safe batch 512, workers 4")

In [ ]:
import subprocess, sys, urllib.request
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pandas", "pyarrow", "torchvision"], check=True)
from google.colab import drive  # type: ignore
drive.mount("/content/drive")
dst = Path("/content/panel_simclr_colab.py")
if dst.exists(): dst.unlink()
urllib.request.urlretrieve(RAW + "panel_simclr_colab.py?v=4", dst)
sys.path.insert(0, "/content")
import torch
assert torch.cuda.is_available(), "needs a CUDA GPU runtime"
print(torch.cuda.get_device_name(0), "— SESSION 0 ready")

In [ ]:
import importlib, sys
import panel_simclr_colab as sim
importlib.reload(sim)
sys.argv = ["simclr", "pretrain", "--drive-panels", DRIVE_PANELS, "--out", OUT_DIR]
sim.main()

In [ ]:
import importlib, sys
import panel_simclr_colab as sim
importlib.reload(sim)
sys.argv = ["simclr", "embed", "--drive-panels", DRIVE_PANELS, "--out", OUT_DIR, "--shard", str(SHARD), "--nshards", str(NSHARDS), "--embed-batch", str(EMBED_BATCH), "--workers", str(WORKERS)]
sim.main()

In [ ]:
from pathlib import Path
combined = Path(OUT_DIR) / "emb_panels_manifest.json"
print("JUDGE INPUT READY" if combined.exists() else "SESSION 0 SHARD SAVED; other shard(s) still pending")
from google.colab import runtime  # type: ignore
runtime.unassign()